# 🥇 Gold Layer — Crypto Business KPIs

**Proyecto:** CoinMarketCap Analytics Platform  
**Capa:** Gold  
**Fuente:** Tabla Silver `crypto_project.silver_crypto_markets_clean`  
**Responsabilidad:** Crear KPIs de negocio listos para análisis, dashboards y toma de decisiones.

**Regla Gold:** los datos ya están limpios y ahora se convierten en métricas útiles para usuarios de negocio.

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    current_timestamp,
    round as spark_round,
    sum as spark_sum,
    avg as spark_avg,
    count as spark_count,
    max as spark_max,
    min as spark_min
)

from pyspark.sql.window import Window

In [0]:
spark.sql("USE SCHEMA crypto_project")

In [0]:
silver_df = spark.table("crypto_project.silver_crypto_markets_clean")

display(
    silver_df.select(
        "coin_id",
        "symbol",
        "coin_name",
        "current_price_usd",
        "market_cap_usd",
        "market_cap_rank",
        "total_volume_usd",
        "price_change_percentage_24h"
    ).orderBy("market_cap_rank")
)

In [0]:
market_window = Window.partitionBy()

gold_market_df = (
    silver_df
    .filter(col("market_cap_rank").isNotNull())
    .withColumn(
        "market_cap_share_top10_pct",
        spark_round(
            (col("market_cap_usd") / spark_sum("market_cap_usd").over(market_window)) * 100,
            2
        )
    )
    .withColumn(
        "volume_to_market_cap_pct",
        spark_round(
            (col("total_volume_usd") / col("market_cap_usd")) * 100,
            2
        )
    )
    .withColumn(
        "intraday_range_pct",
        spark_round(
            ((col("high_24h_usd") - col("low_24h_usd")) / col("current_price_usd")) * 100,
            2
        )
    )
    .withColumn(
        "supply_issued_pct",
        spark_round(
            (col("circulating_supply") / col("max_supply")) * 100,
            2
        )
    )
    .withColumn(
        "performance_signal",
        when(col("price_change_percentage_24h") >= 1, "Positive")
        .when(col("price_change_percentage_24h") <= -1, "Negative")
        .otherwise("Neutral")
    )
    .withColumn(
        "volatility_bucket",
        when(col("intraday_range_pct") >= 5, "High Volatility")
        .when(col("intraday_range_pct") >= 2, "Medium Volatility")
        .otherwise("Low Volatility")
    )
    .withColumn(
        "market_cap_tier",
        when(col("market_cap_rank") <= 2, "Tier 1 - Market Leaders")
        .when(col("market_cap_rank") <= 5, "Tier 2 - Large Cap")
        .otherwise("Tier 3 - Top 10 Assets")
    )
    .withColumn(
        "supply_status",
        when(col("max_supply").isNull(), "No max supply reported")
        .when(col("supply_issued_pct") >= 95, "Almost fully issued")
        .when(col("supply_issued_pct") >= 70, "Mostly issued")
        .otherwise("Still expanding supply")
    )
    .withColumn(
        "gold_processed_ts",
        current_timestamp()
    )
    .select(
        "coin_id",
        "symbol",
        "coin_name",
        "market_cap_rank",
        "market_cap_tier",
        "current_price_usd",
        "market_cap_usd",
        "market_cap_share_top10_pct",
        "total_volume_usd",
        "volume_to_market_cap_pct",
        "high_24h_usd",
        "low_24h_usd",
        "intraday_range_pct",
        "price_change_percentage_24h",
        "performance_signal",
        "volatility_bucket",
        "circulating_supply",
        "max_supply",
        "supply_issued_pct",
        "supply_status",
        "last_updated_ts",
        "ingestion_ts",
        "silver_processed_ts",
        "gold_processed_ts"
    )
)

display(gold_market_df.orderBy("market_cap_rank"))

In [0]:
gold_market_table = "crypto_project.gold_crypto_market_snapshot"

gold_market_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_market_table)

print(f"Tabla Gold creada correctamente: {gold_market_table}")

In [0]:
gold_market_df.createOrReplaceTempView("gold_market_snapshot_temp")

gold_summary_df = spark.sql("""
SELECT
    COUNT(*) AS coins_analyzed,
    ROUND(SUM(market_cap_usd), 2) AS total_market_cap_top10_usd,
    ROUND(AVG(price_change_percentage_24h), 2) AS avg_price_change_24h_pct,
    ROUND(AVG(intraday_range_pct), 2) AS avg_intraday_range_pct,
    ROUND(MAX(CASE WHEN symbol = 'btc' THEN market_cap_share_top10_pct END), 2) AS bitcoin_share_top10_pct,
    ROUND(MAX(CASE WHEN symbol = 'eth' THEN market_cap_share_top10_pct END), 2) AS ethereum_share_top10_pct,
    MAX(gold_processed_ts) AS gold_processed_ts
FROM gold_market_snapshot_temp
""")

display(gold_summary_df)

In [0]:
gold_summary_table = "crypto_project.gold_crypto_market_summary"

gold_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_summary_table)

print(f"Tabla Gold Summary creada correctamente: {gold_summary_table}")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM crypto_project.gold_crypto_market_snapshot
    ORDER BY market_cap_rank ASC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM crypto_project.gold_crypto_market_summary
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY crypto_project.gold_crypto_market_snapshot
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)

In [0]:
display(
    spark.sql("""
    SELECT
        coin_name,
        symbol,
        market_cap_rank,
        market_cap_share_top10_pct,
        market_cap_tier
    FROM crypto_project.gold_crypto_market_snapshot
    ORDER BY market_cap_share_top10_pct DESC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        coin_name,
        symbol,
        price_change_percentage_24h,
        performance_signal
    FROM crypto_project.gold_crypto_market_snapshot
    ORDER BY price_change_percentage_24h DESC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        coin_name,
        symbol,
        intraday_range_pct,
        volatility_bucket
    FROM crypto_project.gold_crypto_market_snapshot
    ORDER BY intraday_range_pct DESC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        coin_name,
        symbol,
        total_volume_usd,
        market_cap_usd,
        volume_to_market_cap_pct
    FROM crypto_project.gold_crypto_market_snapshot
    ORDER BY volume_to_market_cap_pct DESC
    """)
)